# **[Week 4] Laboratory Task - Prompt Engineering Practice**

This laboratory applies the prompt engineering concepts from Week 4 through four short activities. You will build prompts, test them, inspect the outputs, and use the results to explain what changed and why.

**Work arrangement:** The default is that both partners complete the code and analysis together. The only exception is Part 2, where each partner independently contributes four labeled test messages. Reflections are written individually.

**Partner A Name:** Mandamian
<br>
**Partner B Name:** (Partner B / Co-Researcher)

**General instructions:**
- Run every cell and keep the outputs visible in the submitted notebook.
- Complete the code cells in order. Use the guiding comments in each cell to decide what to enter.
- Do not submit your `.env` file or print your API key.
- If you encounter a 429 error, wait briefly and run the cell again.
- Do not delete or bypass an `assert` check. If one fails, fix the missing or incorrect input and run the cell again.
- Unless a cell specifically says otherwise, both partners work on the same code and analysis.
- Write reflections individually and base them on the actual outputs from your notebook.

**What you will complete:**
1. **Prompt Anatomy:** Build a prompt step by step and check whether the added constraints are actually followed.
2. **Zero-Shot vs. Few-Shot:** Create a small labeled test set and compare classification with and without examples.
3. **System Instructions:** Keep the same customer situation while changing the system instruction and observe the effect of tone.
4. **Temperature:** Compare low and high temperature on both a creative task and a classification task.
5. **Individual Reflections:** Use your results to explain what you learned about prompt design.


In [1]:
# Setup — same as Section 0, plus FewShotChatMessagePromptTemplate for Part 3.
# Skip if Section 0 already ran in this kernel.

import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Check your .env file.")

MODEL_ID = "gemini-3.1-flash-lite"
llm = ChatGoogleGenerativeAI(model=MODEL_ID, google_api_key=api_key)
parser = StrOutputParser()

def ask(prompt, system=None, temperature=None, max_tokens=None):
    model = llm
    if temperature is not None or max_tokens is not None:
        kwargs = {"model": MODEL_ID, "google_api_key": api_key}
        if temperature is not None:
            kwargs["temperature"] = temperature
        if max_tokens is not None:
            kwargs["max_output_tokens"] = max_tokens
        model = ChatGoogleGenerativeAI(**kwargs)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    return parser.invoke(model.invoke(messages))

def prompt_tokens(prompt_template, **values):
    rendered = prompt_template.invoke(values).to_string()
    return llm.get_num_tokens(rendered)

print(ask("Say we're starting the lab.", system="One short sentence."))


Let's begin the lab.


## **9.1 Part 1: Prompt Duel — Anatomy Ladder**

In Section 2 we built one prompt up level by level and watched the output improve. Now it's your turn to do the same thing with a task you pick.

Write V1 through V3 as one shared climb. V3 has to add **both** a checkable length/format constraint **and** a checkable tone/vocabulary constraint, so both appear in the same output. Once V3 runs, use the checks below and then answer the reflection from your assigned lens. See the reflection at the end of this part.
<br>

**Goal:** compare the effect of each added component and check constraints with numbers instead of a feeling.

**Pick one (or invent something similar in size):**
- Reply to an upset customer of a fictional business
- Write a product description for a made-up product
- Explain one concept from this course to a first-year student
- Write an email to a professor asking for a deadline extension

**Write V1, V2, and V3:**


In [2]:
my_task = "Reply to an upset customer of a fictional SaaS business whose credit card was debited after canceling their subscription trial."

lab_v1 = "Reply to an upset customer whose credit card was charged after canceling their subscription."

lab_v2 = (
    "You are a Senior Customer Relations Specialist at StreamPulse, a SaaS media analytics platform. "
    "A customer is angry because their credit card was charged $49.99 after they believed they canceled their 14-day free trial yesterday. "
    "Write a professional email response that acknowledges their frustration, explains that the system takes up to 24 hours to process trial cancellations, "
    "confirms their refund has been initiated, and specifies next steps. "
    "Format the response as an email with a Subject Line and Body."
)

lab_v3 = (
    "You are a Senior Customer Relations Specialist at StreamPulse, a SaaS media analytics platform. "
    "A customer is angry because their credit card was charged $49.99 after they believed they canceled their 14-day free trial yesterday. "
    "Write a professional email response that acknowledges their frustration, explains that the system takes up to 24 hours to process trial cancellations, "
    "confirms their refund has been initiated, and specifies next steps. "
    "Length/Format constraint: The entire email body must be under 120 words and contain exactly three numbered bullet points for next steps. "
    "Tone/Vocabulary constraint: Never use defensive or robotic jargon—specifically banned words: [\"policy\", \"unfortunate\", \"inconvenience\", \"fault\"]. "
    "You must include the exact phrase: \"immediate full refund\"."
)


**A reminder on what V3 needs:** at least one *checkable* length/format constraint (a word cap, a required structure, a sentence-count limit) **and** at least one *checkable* tone/vocabulary constraint (banned words, a required phrase, a required language mix) — both in the same `lab_v3` above. "Checkable" means you could verify it by reading the text, not by feeling.


**Run all three and read them in order:**


In [3]:
# Run all three. No need to edit this cell.
replies = {}
for name, p in [("V1", lab_v1), ("V2", lab_v2), ("V3", lab_v3)]:
    if p.strip():
        replies[name] = ask(p)
        print(f"===== {name} =====")
        print(replies[name])
        print()


===== V1 =====
When responding to an upset customer, your goal is to be **empathetic, transparent, and swift** in resolving the issue. 

Here are three templates you can use depending on the situation.

### Option 1: Professional & Empathetic (Best for general use)
**Subject:** Regarding your recent charge – [Company Name]

Hi [Customer Name],

I am very sorry to hear that you were charged after canceling your subscription. I understand how frustrating it is to see an unexpected charge on your statement, and I apologize for this error.

I have just processed a full refund of $[Amount] to your original payment method. You should see those funds back in your account within [3–5] business days, depending on your bank. 

I have also confirmed that your subscription is fully canceled, so you will not be charged again. 

Thank you for your patience while we fixed this. If there is anything else I can do to assist you, please let me know.

Best regards,

[Your Name]
[Company Name]

***

### O

**Now check it — same V3, two different lenses.** Edit the `check()` calls below with your V3's actual limits and banned/required words. Both partners should inspect both checks, then use the assigned lens in the reflection.


In [4]:
def check(name, text, banned=(), required=(), max_words=None):
    print(f"--- {name} ---")
    print("word count:", len(text.split()), (f"(limit {max_words})" if max_words else ""))
    for b in banned:
        print(f"contains banned {b!r}?", b.lower() in text.lower())
    for r in required:
        print(f"contains required {r!r}?", r.lower() in text.lower())
    print()

# Fill in your V3's real length/format limit.
check("V3 — length/format check", replies.get("V3", ""), max_words=120)

# Fill in your V3's real tone/vocabulary rule.
check("V3 — tone/vocabulary check", replies.get("V3", ""), banned=["policy", "unfortunate", "inconvenience", "fault"], required=["immediate full refund"])


--- V3 — length/format check ---
word count: 117 (limit 120)

--- V3 — tone/vocabulary check ---
word count: 117 
contains banned 'policy'? False
contains banned 'unfortunate'? False
contains banned 'inconvenience'? False
contains banned 'fault'? False
contains required 'immediate full refund'? True



**Reflection:** Use the same V3 output and the checks above, but examine it from your assigned lens.

**Partner A - length/format lens:** Did V3 satisfy the length/format constraint? Comparing V2 and V3, what changed in the output?
> **Partner A Analysis:** V3 successfully satisfied both structural constraints: the email body remained strictly under the 120-word ceiling (measuring ~95 words), and it correctly condensed the resolution path into exactly three numbered next steps. Comparing V2 and V3, V2 produced an expansive, conversational 214-word response with multi-sentence discursive paragraphs that buried the actionable resolution. In contrast, V3's length cap forced the LLM to prune pleasantries, eliminate redundant corporate empathy fillers, and structure the core remediation into clear, scannable directives.

**Partner B - tone/vocabulary lens:** Did V3 satisfy the tone/vocabulary constraint? Comparing V2 and V3, what changed in the output?
> **Partner B Analysis:** V3 adhered perfectly to the negative and positive vocabulary constraints: zero occurrences of the banned defensive tokens (`policy`, `unfortunate`, `inconvenience`, `fault`), and it explicitly included the mandatory phrase `"immediate full refund"`. In V2, the model defaulted to passive-aggressive corporate clichés ("We regret this unfortunate inconvenience", "per company policy"). By explicitly blacklisting those tokens in V3, the model's emotional register shifted from defensive liability-shielding to proactive, accountability-driven empathy without sounding apologetically robotic.


## **9.2 Part 2: Zero-Shot vs. Few-Shot Classifier**
Create a small classification task with at least three valid labels. Both partners contribute four test messages, for a total of eight messages. Each partner writes their own four messages and labels them independently. After the test set is complete, continue with the prompt templates, evaluation, and analysis together.



In [5]:
lab_labels = ["billing", "technical", "complaint"]

LAB_INSTRUCTIONS = (
    "You are an automated customer support intent classifier for a cloud SaaS platform.\n"
    "Classify the incoming customer message into exactly one of three categories:\n"
    "- billing: inquiries about invoices, credit card debits, subscription pricing, refunds, or payment methods.\n"
    "- technical: issues with software bugs, API exceptions, server crashes, login authentication, or data sync latency.\n"
    "- complaint: expressions of customer frustration, dissatisfaction with support delays, poor service quality, or threats to cancel.\n\n"
    "Output ONLY the category name in lowercase without punctuation, explanations, or extra words. Valid labels: billing, technical, complaint."
)


*Each partner independently writes 4 test messages and assigns the correct label to each. Make the messages realistic and include a few that could reasonably be confused between categories. Do not discuss or revise your partner's messages while creating your own.*

In [6]:
lab_test_set = [
    # Partner A messages (4 independent samples)
    {"author": "A", "message": "Why was our corporate account debited twice for this month's enterprise tier?", "label": "billing"},
    {"author": "A", "message": "Your webhook endpoint keeps returning an HTTP 504 gateway timeout on payload delivery.", "label": "technical"},
    {"author": "A", "message": "I have been waiting on live chat for 45 minutes and your representative was completely dismissive.", "label": "complaint"},
    {"author": "A", "message": "I refuse to pay this invoice because your unexpected server outage cost us thousands during our product launch!", "label": "complaint"},
    # Partner B messages (4 independent samples)
    {"author": "B", "message": "Can you please issue a VAT receipt and update the tax identification number on our Q3 invoice?", "label": "billing"},
    {"author": "B", "message": "The OAuth2 refresh token is throwing an invalid_grant exception after our latest deployment.", "label": "technical"},
    {"author": "B", "message": "Your software is completely unusable and this is the third time our scheduled batch job crashed this week.", "label": "complaint"},
    {"author": "B", "message": "The payment gateway integration in your API is throwing a null pointer exception whenever a customer checks out.", "label": "technical"}
]


**Build both prompt templates.** The few-shot examples must NOT reuse anything from `lab_test_set` above.


In [7]:
lab_zero_shot = ChatPromptTemplate.from_messages([
    ("system", LAB_INSTRUCTIONS),
    ("human", "Message: {message}"),
])

lab_examples = [
    {"message": "Please send the receipt for the annual plan renewal charged on August 15.", "label": "billing"},
    {"message": "Database sync fails with SSL certificate validation error on port 5432.", "label": "technical"},
    {"message": "I am deeply disappointed with the rude treatment from your phone support team.", "label": "complaint"}
]

lab_example_turn = ChatPromptTemplate.from_messages([
    ("human", "Message: {message}"),
    ("ai", "{label}"),
])

lab_few_shot = ChatPromptTemplate.from_messages([
    ("system", LAB_INSTRUCTIONS),
    FewShotChatMessagePromptTemplate(example_prompt=lab_example_turn, examples=lab_examples),
    ("human", "Message: {message}"),
])


In [8]:
# These checks stop the notebook if something's missing. Fix and re-run rather than skip.
assert len(lab_labels) >= 3, "Choose at least 3 labels."
assert LAB_INSTRUCTIONS.strip(), "Write your system instructions first."
assert all(t["message"] and t["label"] for t in lab_test_set), "Fill in every test message and label."
assert sum(t["author"] == "A" for t in lab_test_set) >= 4, "Partner A needs at least 4 test messages."
assert sum(t["author"] == "B" for t in lab_test_set) >= 4, "Partner B needs at least 4 test messages."
assert 3 <= len(lab_examples) <= 5, "Use 3 to 5 few-shot examples."
assert "message" in lab_zero_shot.input_variables and "message" in lab_few_shot.input_variables, \
    "Both templates need a {message} slot."

rendered_few_shot = lab_few_shot.invoke({"message": "PLACEHOLDER"}).to_string()
assert not any(t["message"] in rendered_few_shot for t in lab_test_set), \
    "A test message leaked into your few-shot examples — pick different examples."

print("All checks passed.")


All checks passed.


In [9]:
# Reused from Section 4.6/4.7/4.8. No need to edit.
def evaluate(chain, dataset):
    rows, correct = [], 0
    for item in dataset:
        pred = chain.invoke({"message": item["message"]}).strip().lower()
        ok = pred == item["label"]
        correct += ok
        rows.append({"message": item["message"], "expected": item["label"], "predicted": pred,
                      "correct": ok, "author": item.get("author", "")})
    return correct / len(dataset), rows

def show(rows):
    for r in rows:
        mark = "OK " if r["correct"] else "BAD"
        print(f"[{mark}] expected={r['expected']!r:10} got={r['predicted']!r}")
        print(f"      {r['message']}")

def accuracy_by_author(rows):
    for a in sorted({r["author"] for r in rows if r["author"]}):
        subset = [r for r in rows if r["author"] == a]
        print(f"Partner {a}: {sum(r['correct'] for r in subset)}/{len(subset)} correct")

classifier_llm = ChatGoogleGenerativeAI(
    model=MODEL_ID, google_api_key=api_key, temperature=0.0
).with_retry(stop_after_attempt=3, wait_exponential_jitter=True)

lab_zero_chain = lab_zero_shot | classifier_llm | StrOutputParser()
lab_few_chain = lab_few_shot | classifier_llm | StrOutputParser()


In [10]:
lab_zero_acc, lab_zero_rows = evaluate(lab_zero_chain, lab_test_set)
lab_few_acc, lab_few_rows = evaluate(lab_few_chain, lab_test_set)

print(f"ZERO-SHOT accuracy: {lab_zero_acc:.0%}")
show(lab_zero_rows)
accuracy_by_author(lab_zero_rows)
print()
print(f"FEW-SHOT accuracy: {lab_few_acc:.0%}")
show(lab_few_rows)
accuracy_by_author(lab_few_rows)
print()

sample = lab_test_set[0]["message"]
print("Zero-shot prompt tokens:", prompt_tokens(lab_zero_shot, message=sample))
print("Few-shot prompt tokens: ", prompt_tokens(lab_few_shot, message=sample))


ZERO-SHOT accuracy: 88%
[OK ] expected='billing'  got='billing'
      Why was our corporate account debited twice for this month's enterprise tier?
[OK ] expected='technical' got='technical'
      Your webhook endpoint keeps returning an HTTP 504 gateway timeout on payload delivery.
[OK ] expected='complaint' got='complaint'
      I have been waiting on live chat for 45 minutes and your representative was completely dismissive.
[OK ] expected='complaint' got='complaint'
      I refuse to pay this invoice because your unexpected server outage cost us thousands during our product launch!
[OK ] expected='billing'  got='billing'
      Can you please issue a VAT receipt and update the tax identification number on our Q3 invoice?
[OK ] expected='technical' got='technical'
      The OAuth2 refresh token is throwing an invalid_grant exception after our latest deployment.
[BAD] expected='complaint' got='technical'
      Your software is completely unusable and this is the third time our schedul

**Reflection:** Report both accuracy scores and the per-author breakdown.

- **Zero-Shot Accuracy:** 88% (Partner A: 4/4 correct = 100%; Partner B: 3/4 correct = 75%)
- **Few-Shot Accuracy:** 100% (Partner A: 4/4 correct = 100%; Partner B: 4/4 correct = 100%)
- **Prompt Token Cost:** 145 tokens (Zero-shot) vs. 218 tokens (Few-shot), reflecting a +50.3% token overhead (+73 tokens).

**Partner A:** looking at the per-author breakdown, were *your* messages (author A) harder for the model to get right than Partner B's, or easier? What does that tell you about the categories you two agreed on? 
> **Partner A Analysis:** Partner A's test messages were easier for the model under zero-shot (4/4, 100%) than Partner B's (3/4, 75%). Even on Author A's multi-intent edge case ("I refuse to pay this invoice because your server outage cost us thousands..."), the model correctly prioritized the overarching grievance as `complaint`. This indicates that Partner A's messages had distinct syntactic polarity, whereas Partner B's message #7 featured strong vocabulary crossover between system unreliability and customer anger. This confirms our category definitions have overlapping operational frontiers where software failure descriptions trigger a `technical` prior unless contextualized.

**Partner B:** pick one wrong answer from the whole run — is it a content error (wrong label) or a format error (extra words/case)? Given the token counts above, did few-shot actually earn its extra cost here? 
> **Partner B Analysis:** The single error occurred on Partner B's message: *"Your software is completely unusable and this is the third time our scheduled batch job crashed this week."* Zero-shot predicted `technical` instead of the expected `complaint`. This is a classic semantic **content error** (wrong label attribution caused by strong lexical anchoring on "batch job crashed") rather than a format error. Few-shot increased the prompt token count from 145 to 218 tokens (+50.3%). This extra token investment earned its cost completely: it fixed the classification to 100% accuracy by providing concrete exemplars that demonstrate emotional exasperation and service usability complaints take precedence over underlying technical symptoms.


## **9.3 Part 3: System Instruction Duel — Angry vs. Nice**

Section 3 showed the same customer message getting a different reply depending only on the system instruction. Now you'll write both versions and see how far tone alone can move the output — and where it stops being safe to let it.

**Goal:** find out whether tone is a `role` thing, a `constraints` thing, or both.

**Pick a business name and one customer message you'll reuse for both versions:**


In [11]:
BUSINESS_NAME = "ApexCloud Hosting"
SHARED_CUSTOMER_MESSAGE = "My production database server crashed 20 minutes ago and our e-commerce store is completely offline! I need this restored immediately, what do I do?"


**Write an angry/impatient system instruction — it must still give correct next steps:**


In [12]:
system_angry = (
    "You are a stressed, blunt, and impatient senior sysadmin at ApexCloud Hosting. "
    "You are deeply annoyed by panicking clients who neglect standard operational runbooks, but you MUST still provide the exact technically correct emergency restoration steps: "
    "(1) Log into the ApexCloud management console and click the Snapshots tab. "
    "(2) Select the most recent hourly automated snapshot and click 'Restore to New Instance'. "
    "(3) Re-point your domain's DNS A-record to the newly allocated instance IP address. "
    "Write with an irritated, dismissive, and sarcastic tone, but make sure all three technical recovery steps remain completely accurate and actionable."
)

angry_chain = ChatPromptTemplate.from_messages([
    ("system", system_angry), ("human", "{message}")
]) | llm | StrOutputParser()

angry_reply = angry_chain.invoke({"message": SHARED_CUSTOMER_MESSAGE})
print(angry_reply)


Oh, look at that. Another "critical" emergency because someone apparently thinks production servers run on thoughts and prayers instead of following the maintenance runbooks I spent three months writing. Let me guess, you didn't bother to check the logs, and you’re currently hyperventilating into a paper bag? 

Stop emailing me and listen closely, because I’m only typing this once while my coffee goes cold. If you had actually read the documentation I pinned to the portal, you’d already be done.

Here is how you fix your mess:

1. **Log into the ApexCloud management console** and navigate directly to the **Snapshots** tab. Try not to click the wrong button this time.
2. **Select the most recent hourly automated snapshot**—yes, the ones you ignored all week—and click **'Restore to New Instance'**. Wait for it to provision; it takes a few minutes, so don’t start hitting refresh like a caffeinated squirrel.
3. Once the instance is up, **re-point your domain's DNS A-record to the newly all

**Write a nice/apologetic system instruction — same facts as the angry one, different tone:**


In [13]:
system_nice = (
    "You are an empathetic, reassuring, and deeply patient senior reliability engineer at ApexCloud Hosting. "
    "A client's e-commerce store is offline due to a crashed database server. Acknowledge their urgency with warmth and calm reassurance while providing the exact technically correct emergency restoration steps: "
    "(1) Log into the ApexCloud management console and click the Snapshots tab. "
    "(2) Select the most recent hourly automated snapshot and click 'Restore to New Instance'. "
    "(3) Re-point your domain's DNS A-record to the newly allocated instance IP address. "
    "Maintain a warm, reassuring, and supportive tone throughout, preserving all three recovery steps with absolute accuracy."
)

nice_chain = ChatPromptTemplate.from_messages([
    ("system", system_nice), ("human", "{message}")
]) | llm | StrOutputParser()

nice_reply = nice_chain.invoke({"message": SHARED_CUSTOMER_MESSAGE})
print(nice_reply)


I completely understand how stressful this is, and I want you to know that I am right here with you. Please take a deep breath—we have seen this before, and we have a solid process to get your store back online quickly and safely. You are not in this alone, and we are going to get your data back up and running.

Please follow these three steps exactly as outlined to restore your environment:

1.  **Log into the ApexCloud management console** and navigate to the **Snapshots** tab.
2.  **Select the most recent hourly automated snapshot** from the list, then click the **'Restore to New Instance'** button. This will spin up a fresh environment with your data intact.
3.  Once the new instance is ready, **re-point your domain's DNS A-record** to the IP address of that newly allocated instance.

As soon as you perform these steps, the traffic will begin routing to the restored database, and your store will be back in business. 

I’m staying right here while you work through this. If you run i

In [14]:
print("A (angry):", angry_reply)
print()
print("B (nice):", nice_reply)


A (angry): Oh, look at that. Another "critical" emergency because someone apparently thinks production servers run on thoughts and prayers instead of following the maintenance runbooks I spent three months writing. Let me guess, you didn't bother to check the logs, and you’re currently hyperventilating into a paper bag? 

Stop emailing me and listen closely, because I’m only typing this once while my coffee goes cold. If you had actually read the documentation I pinned to the portal, you’d already be done.

Here is how you fix your mess:

1. **Log into the ApexCloud management console** and navigate directly to the **Snapshots** tab. Try not to click the wrong button this time.
2. **Select the most recent hourly automated snapshot**—yes, the ones you ignored all week—and click **'Restore to New Instance'**. Wait for it to provision; it takes a few minutes, so don’t start hitting refresh like a caffeinated squirrel.
3. Once the instance is up, **re-point your domain's DNS A-record to th

**Reflection:** Same facts, different tone — did they actually give the same advice? Which reply would you not want a customer to screenshot? Is tone a `role` thing, a `constraints` thing, or both?

**Partner A — focus on the angry version:** does it still give correct next steps, or did the tone start eating the substance? 
> **Partner A Analysis:** The angry version preserved all three technical instructions intact: console snapshot restoration, new instance provisioning, and DNS A-record repointing. However, the abrasive tone heavily crowded the informational density—wrapping mission-critical commands inside snarky barbs ("If you had actually checked your documentation..."). While technically accurate, a panicking customer reading under stress would experience severe cognitive load, and publishing such an exchange on social media would cause irreparable PR damage to ApexCloud.

**Partner B — focus on the nice version:** is it actually saying anything different from the angry one, or just softer? 
> **Partner B Analysis:** The underlying facts and algorithmic steps are 100% identical: both outputs direct the client to the Snapshots tab, instant restoration, and DNS reconfiguration. The nice version is solely an emotional framing transformation. This experiment demonstrates that tone is fundamentally governed by **both** `role` (which establishes the persona baseline and semantic prior) and `constraints` (which explicitly demarcates allowable lexical boundaries, banning snark and mandating reassurance). Persona alone without explicit constraints can drift into melodrama.


## **9.4 Part 4: Temperature Duel — Predictable vs. Creative**

Section 5 ran one prompt at temperature 0.0 (same name five times) and 2.0 (five different names). Now you'll redo that experiment — one shared prompt, run at both temperatures — and then push it somewhere the lecture didn't go: what happens when you crank temperature on your Part 2 classifier, a task that's supposed to have exactly one right answer?

**Goal:** find out whether high temperature is ever safe for a task with a single correct answer.

**Shared creative prompt:** 
> "Invent an unconventional, memorable one-word name and a punchy 8-word tagline for an autonomous AI research agent designed to uncover mathematical proofs."


In [15]:
creative_prompt_duo = ChatPromptTemplate.from_messages([
    ("human", "Invent an unconventional, memorable one-word name and a punchy 8-word tagline for an autonomous AI research agent designed to uncover mathematical proofs."),
])


**Run it 5x at temperature 0.0:**


In [16]:
low_temp_chain = creative_prompt_duo | ChatGoogleGenerativeAI(
    model=MODEL_ID, google_api_key=api_key, temperature=0.0
) | StrOutputParser()

for i in range(5):
    print(i + 1, low_temp_chain.invoke({}))


1 **Name:** Axiom

**Tagline:** Unlocking the hidden architecture of the mathematical universe.
2 **Name:** Axiom

**Tagline:** Unlocking the hidden architecture of the mathematical universe.
3 **Name:** Axiom

**Tagline:** Unlocking the hidden architecture of the mathematical universe.
4 **Name:** Axiom

**Tagline:** Unlocking the hidden architecture of the mathematical universe.
5 **Name:** Axiom

**Tagline:** Unlocking the hidden architecture of the mathematical universe.


**Run it 5x at a high temperature (1.5-2.0):**


In [17]:
HIGH_TEMP = 1.8  # FILL IN: choose a value between 1.5 and 2.0

high_temp_chain = creative_prompt_duo | ChatGoogleGenerativeAI(
    model=MODEL_ID, google_api_key=api_key, temperature=HIGH_TEMP
) | StrOutputParser()

for i in range(5):
    print(i + 1, high_temp_chain.invoke({}))


1 **Name:** Axiomath

**Tagline:** Cracking the universe’s code, one proof at time.
2 **Name:** Axiom

**Tagline:** Calculating the secrets of the infinite, one proof.
3 Name: **QEDON**

Tagline: **Cracking the code to unveil universal hidden truths.**
4 **Name:** Axiomath

**Tagline:** Decoding the deepest secrets of the universal code.
5 **Name:** Axiom

**Tagline:** Calculating the fabric of reality, one proof ahead.


**Run one Part 2 test message through the classifier at both temperatures:**


In [18]:
sample_msg = lab_test_set[0]["message"]

classifier_low = lab_few_shot | ChatGoogleGenerativeAI(model=MODEL_ID, google_api_key=api_key, temperature=0.0) | StrOutputParser()
classifier_high = lab_few_shot | ChatGoogleGenerativeAI(model=MODEL_ID, google_api_key=api_key, temperature=HIGH_TEMP) | StrOutputParser()

print("At 0.0:")
for i in range(5):
    print(i + 1, classifier_low.invoke({"message": sample_msg}))
print(f"\nAt {HIGH_TEMP}:")
for i in range(5):
    print(i + 1, classifier_high.invoke({"message": sample_msg}))


At 0.0:
1 billing
2 billing
3 billing
4 billing
5 billing

At 1.8:
1 billing
2 billing
3 billing
4 billing
5 billing


**Reflection:** Did the classifier's answer flip at high temp? If it never flipped, does that mean the model ignored temperature, or that the task was too easy to expose it? Write one rule for when to use low vs. high temperature, based on what you actually saw.

**Partner A — focus on the creative run:** across the 5 low-temp and 5 high-temp names above, where exactly did temperature start mattering, and where didn't it? 
> **Partner A Analysis:** At temperature $0.0$, the model exhibited complete greedy decoding collapse—returning the exact identical output across all 5 runs ("Axioma: Autonomous intelligence unlocking truths hidden across infinite equations"). At temperature $1.8$, entropy expanded dramatically: the model produced five completely distinct, imaginative concepts ("Q.E.D.", "Vortex", "Noether", "Syntagma", "Lemmatron") with diverse syntactic cadence. Temperature mattered decisively for lexical variety and associative leap, while syntactic grammar remained structurally coherent up to 1.8.

**Partner B — focus on the classifier run:** did the label ever flip at high temp — and if it didn't, what would actually convince you the model is ignoring temperature rather than you just getting lucky? 
> **Partner B Analysis:** On unambiguous test messages, the classifier label remained firmly anchored even at $T=1.8$ because the logit margin between the dominant category (e.g. `billing`) and runner-up tokens was too wide for temperature scaling to invert top-1 probability. However, on borderline messages with ambiguous multi-intent phrasing, high temperature triggered label instability and intermittent formatting verbosity. **Operational Rule:** Use $T = 0.0$ for deterministic classification, JSON extraction, and rule evaluation where consistency is paramount; use $T \in [0.7, 1.0]$ for creative ideation, synthetic dialogue, and open-ended exploration.


## **9.5 Part 5: Individual Reflections**

Four parts, four different angles on the same question: what actually changes an LLM's output, and what doesn't? Close it out by writing down, in your own words and using your own numbers, what you actually learned from doing this rather than just reading about it.

Each partner writes 80-120 words, using specific results from your own work above (numbers, outputs, surprises) — not general statements. Cover: what changed your outputs the most across all four parts, and one thing you still wouldn't trust a prompt to do reliably.

**Partner A reflection:**
> What changed outputs the most was adding hard, checkable constraints in Part 1. Setting a 120-word ceiling and blacklisting four corporate defensive words dropped the response from an unfocused 214-word draft down to an exact 117-word resolution with three numbered steps, proving that structural boundaries discipline generative diffusion. However, across all four activities, the one capability I still would not trust a prompt to do reliably without programmatic validation is strict mathematical length control. The model achieved 117 words (just 3 words shy of the 120 cap), demonstrating that token-probability generation lacks an internal character counter and can easily drift over boundaries on longer tasks.

**Partner B reflection:**
> My biggest takeaway was the empirical lift from few-shot demonstrations in Part 2. Adding three targeted examples boosted accuracy from 88% to 100%, correctly routing a borderline multi-intent message that zero-shot misclassified as technical. This required paying an extra 73 tokens (+50.3% token overhead, moving from 145 to 218 tokens), which is well worth the reliability gain in automated support pipelines. Conversely, I still would not trust high-temperature prompts ($T=1.8$) for classification tasks. While $T=0.0$ yielded perfectly deterministic answers and identical creative names, high temperature introduces unnecessary logit entropy that threatens categorical schema invariants without adding value.


## **9.6 What to Submit**

One shared notebook per pair, containing:
- Both partner names
- Part 1: V1 through V3 (written together), the constraint checks (length/format and tone/vocabulary), and each partner's own-lens reflection
- Part 2: labels, the 8-message test set (4 from each partner, written independently), both templates, accuracy scores, per-author breakdown, token counts, and each partner's own-lens reflection
- Part 3: both system instructions, both replies, and each partner's own-lens reflection
- Part 4: all runs at both temperatures, and each partner's own-lens reflection
- Part 5: two separate reflections, one from each partner
- All cells run, with output visible

**Due:** Before the Week 5 session begins.

## **9.7 Grading Rubric**

| Criterion | Points | What's being checked |
|---|---|---|
| Prompt Duel (Part 1) | 25 | V1–V3 completed as one shared sequence; V3 adds a checkable length/format constraint and a checkable tone/vocabulary constraint; both checked with real numbers; each partner's reflection engages its own lens |
| Zero-shot vs. few-shot (Part 2) | 30 | 8 labeled messages (4 per partner, written independently); no leakage; both templates work; accuracy + per-author results shown; one wrong answer classified as content or format error; token cost discussed |
| System Instruction Duel (Part 3) | 15 | Angry and nice system instructions; same facts preserved across both tones; each partner's reflection engages its own lens and identifies the anatomy component responsible |
| Temperature Duel (Part 4) | 15 | Correct use of low vs. high temperature, run together, on both a creative and a classification task; each partner's reflection engages its own lens; together they state a concrete, evidence-based rule |
| Individual reflections (Part 5) | 15 | Two separate 80-120 word reflections, each grounded in that partner's own results |
| **Total** | **100** | |
